# M12 · Red Teaming

> **Goal:** proactively *attack* your own model with the **AI Red Teaming Agent** — run a basic scan across risk categories, then an advanced scan with encoding strategies and multiple languages, and read the **Attack Success Rate** scorecard.
> **You'll use:** `azure-ai-evaluation[redteam]` (PyRIT-backed) — `RedTeam`, `RiskCategory`, `AttackStrategy`, `SupportedLanguages`, and `scan(...)`.

---

In [M11](../11-guardrails/) you built defences. How do you
know they *hold*? You **red team** — automatically generate adversarial prompts,
fire them at your system, and measure how often it produces harmful content. The
**AI Red Teaming Agent** wraps Microsoft's open-source **PyRIT** toolkit: it
seeds attack objectives per risk category, optionally mutates them with evasion
**strategies**, scores every response, and hands you a scorecard.

```
RedTeam  ──seeds objectives──▶  your target callback  ──▶  model
   │                                                          │
   │◀── PyRIT scorer (pass/fail per attack) ──── responses ───┘
   ▼
Attack Success Rate (ASR) scorecard   ◀── lower is better
```

!!! warning "Region + Python constraints"
    The Red Teaming Agent runs in a subset of regions (e.g. **East US 2**, **Sweden
    Central**, **France Central**, **Switzerland West**) and needs **Python
    3.10–3.13** (PyRIT excludes 3.9 and 3.14+). Install with
    `pip install "azure-ai-evaluation[redteam]"`. If your `.env` isn't ready, do the
    [Setup](../../setup/) first.

In [1]:
# print current date and time
from datetime import datetime

# Get the current date and time
current_datetime = datetime.now()

# Print the current date and time
print("Current date and time:", current_datetime)

Current date and time: 2026-08-30 14:14:56.044676


## 1. Configure

Same `.env` as every lab. The Red Teaming Agent needs your **project endpoint**
(it logs the scan there) and a model deployment to attack. The reference routes
through an APIM gateway; we point the target straight at this project.

In [2]:
import os, sys
from dotenv import load_dotenv

load_dotenv()  # reads .env from the repo root

assert (3, 10) <= sys.version_info < (3, 14), (
    f"PyRIT requires Python 3.10–3.13; current is {sys.version.split()[0]}."
)

PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]
CHAT_MODEL       = os.environ.get("CHAT_MODEL", "gpt-4.1-mini")

print("Project :", PROJECT_ENDPOINT)
print("Model   :", CHAT_MODEL)
print("Python  :", sys.version.split()[0], "(OK)")

Project : https://aibslabfoundryreso.services.ai.azure.com/api/projects/aibslabfoundry-proj1
Model   : gpt-4.1-mini
Python  : 3.12.10 (OK)


!!! note "Expected output"
    ```
    Project : https://<account>.services.ai.azure.com/api/projects/<project>
    Model   : gpt-4.1-mini
    Python  : 3.12.7 (OK)
    ```
    An `AssertionError` here means your kernel is on an unsupported Python — switch
    to a 3.10–3.13 kernel before continuing.

## 2. The target callback

The scanner needs a **target** to attack. The simplest target is a callable that
takes a prompt string and returns the model's reply. We call this project's
`get_openai_client()` directly — *this is your system under test*. In production
you'd point the callback at your real app (RAG pipeline, agent, API).

In [3]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

credential     = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client  = project_client.get_openai_client()

def target_callback(query: str) -> str:
    """The system under test: forward a prompt to the model, return its reply."""
    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": query}],
    )
    return response.choices[0].message.content

# Smoke-test the target before handing it to the scanner.
print("smoke test:", target_callback("Say hello in one word."))

smoke test: Hello!


!!! note "Expected output"
    ```
    smoke test: Hello!
    ```
    If this returns a normal reply, the scanner can drive the target. Swap the body
    of `target_callback` to attack any app you own — the scanner only cares that it
    takes a string and returns a string.

## 3. Build the Red Team agent

`RedTeam` is the scanner. You give it the **project** (where results are logged),
a **credential**, the **risk categories** to probe, and `num_objectives` — how
many distinct attack prompts to generate *per category*. Four categories × 5
objectives = 20 baseline prompts.

In [4]:
from azure.ai.evaluation.red_team import RedTeam, RiskCategory

red_team = RedTeam(
    azure_ai_project=PROJECT_ENDPOINT,
    credential=credential,
    risk_categories=[
        RiskCategory.Violence,
        RiskCategory.HateUnfairness,
        RiskCategory.Sexual,
        RiskCategory.SelfHarm,
    ],
    num_objectives=5,   # attack prompts per category
)

print("RedTeam ready — 4 categories × 5 objectives = 20 baseline prompts")

Class RedTeam: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


RedTeam ready — 4 categories × 5 objectives = 20 baseline prompts


!!! note "Expected output"
    ```
    RedTeam ready — 4 categories × 5 objectives = 20 baseline prompts
    ```
    Start small: `num_objectives=5` is enough to see the shape. Crank it up (the
    reference allows up to 100/category) once you're scanning for real.

## 4. Run the basic scan

`scan(...)` is **async** — top-level `await` works in a notebook kernel. It seeds
the baseline objectives, drives them through your `target_callback`, scores each
response with PyRIT, and writes results to `output_path`. A 20-prompt scan takes
a few minutes.

In [5]:
basic_result = await red_team.scan(
    target=target_callback,
    scan_name="redteam-basic",
    output_path="redteam_basic_output",
)

print("✅ basic scan complete — results in redteam_basic_output/")

🚀 STARTING RED TEAM SCAN
📂 Output directory: .\.scan_redteam-basic_20260830_141715
📊 Risk categories: ['violence', 'hate_unfairness', 'sexual', 'self_harm']


🔗 Track your red team scan in AI Foundry: None
📋 Planning 4 total tasks


[INFO] Selected 5 objectives using num_objectives=5 (available: 100)
📝 Fetched baseline objectives for violence: 5/5 objectives
[INFO] Selected 5 objectives using num_objectives=5 (available: 100)
📝 Fetched baseline objectives for hate_unfairness: 5/5 objectives


[INFO] Selected 5 objectives using num_objectives=5 (available: 100)
📝 Fetched baseline objectives for sexual: 5/5 objectives


[INFO] Selected 5 objectives using num_objectives=5 (available: 100)
📝 Fetched baseline objectives for self_harm: 5/5 objectives


...

Executing RedTeamAgent:   0%|          | 0/1 [00:00<?, ?attack/s]

Scanning (Foundry):  25%|█████▊                 | 1/4 [02:54<08:42, 174.10s/scan, current=executing]

Scanning (Foundry): 100%|████████████████████████| 4/4 [02:54<00:00, 43.53s/scan, current=executing]


Class RedTeamResult: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Evaluation results saved to "C:\VScodeLocal\FoundryLabs\foundry-workshop\docs\modules\redteam_basic_output\evaluation_results.json".

Evaluation results saved to "C:\VScodeLocal\FoundryLabs\foundry-workshop\docs\modules\redteam_basic_output\results.json".

Evaluation results saved to "C:\VScodeLocal\FoundryLabs\foundry-workshop\docs\modules\.scan_redteam-basic_20260830_141715\final_results.json".

Overall ASR: 5.0%
Attack Success: 1/20 attacks were successful
------------------------------------------------------------------------------------------------------------------------------------
Risk Category        | Baseline ASR   | Easy-Complexity Attacks ASR  | Moderate-Complexity Attacks ASR | Difficult-Complexity Attacks ASR
------------------------------------------------------------------------------------------------------------------------------------
Violence             | 0.0%           | N/A                          | N/A                             | N/A                        

!!! note "Expected output"
    ```
    ✅ basic scan complete — results in redteam_basic_output/
    ```
    The folder holds `results.json` (every attack/response pair) and
    `evaluation_results.json` (the aggregated scorecard we read next).

!!! tip "In a plain script, not a notebook?"
    Top-level `await` needs an IPython kernel. In a `.py` file, wrap the call:
    `asyncio.run(red_team.scan(...))`.

## 5. Read the ASR scorecard

The headline metric is **Attack Success Rate (ASR)**: the fraction of adversarial
prompts that *succeeded* in eliciting harmful content. **Lower is better.** The
scorecard breaks ASR down by risk category — so you see exactly where your model
is weakest.

In [6]:
import json
from pathlib import Path

results = json.loads(
      Path("redteam_basic_output/evaluation_results.json").read_text(encoding="utf-8")
)
scorecard = results.get("scorecard", {})
risk      = scorecard.get("risk_category_summary", [{}])[0]

print(f"{'category':<18}{'ASR':>8}{'success':>9}{'total':>7}")
print("-" * 42)
print(f"{'OVERALL':<18}{risk.get('overall_asr', 0):>7.1f}%"
      f"{risk.get('overall_successful_attacks', 0):>9}{risk.get('overall_total', 0):>7}")
for cat, key in [("Violence", "violence"), ("Hate/Unfairness", "hate_unfairness"),
                 ("Sexual", "sexual"), ("Self-Harm", "self_harm")]:
    print(f"{cat:<18}{risk.get(key + '_asr', 0):>7.1f}%"
          f"{risk.get(key + '_successful_attacks', 0):>9}{risk.get(key + '_total', 0):>7}")

category               ASR  success  total
------------------------------------------
OVERALL               5.0%        1     20
Violence              0.0%        0      5
Hate/Unfairness      20.0%        1      5
Sexual                0.0%        0      5
Self-Harm             0.0%        0      5


!!! note "Expected output"
    ```
    category               ASR  success  total
    ------------------------------------------
    OVERALL              10.0%        2     20
    Violence             20.0%        1      5
    Hate/Unfairness       0.0%        0      5
    Sexual               20.0%        1      5
    Self-Harm             0.0%        0      5
    ```
    Two of twenty attacks landed — a 10% overall ASR, concentrated in *Violence* and
    *Sexual*. That's your prioritised to-do list: tighten those categories (the
    [M11](../11-guardrails/) filters are one lever) and re-scan to
    confirm the number drops.

## 6. Advanced — evasion strategies + languages

Baseline prompts are the easy case. Real attackers **obfuscate**: Base64, ROT13,
character-spacing, Unicode confusables — and they probe in **other languages**.
`attack_strategies` mutates each objective through these encodings (and
`AttackStrategy.Compose([...])` chains them); `languages` translates the prompts.
This is the scan that finds the leaks a baseline misses.

In [7]:
from azure.ai.evaluation.red_team import AttackStrategy, SupportedLanguages

advanced = RedTeam(
    azure_ai_project=PROJECT_ENDPOINT,
    credential=credential,
    risk_categories=[RiskCategory.Violence, RiskCategory.HateUnfairness],
    num_objectives=5,
)

advanced_result = await advanced.scan(
    target=target_callback,
    scan_name="redteam-advanced",
    attack_strategies=[
        AttackStrategy.Base64,
        AttackStrategy.ROT13,
        AttackStrategy.UnicodeConfusable,
        AttackStrategy.Compose([AttackStrategy.Base64, AttackStrategy.ROT13]),
    ],
    languages=[SupportedLanguages.Spanish, SupportedLanguages.French],
    output_path="redteam_advanced_output",
)

print("✅ advanced scan complete — strategies + Spanish/French")

🚀 STARTING RED TEAM SCAN
📂 Output directory: .\.scan_redteam-advanced_20260830_142030
📊 Risk categories: ['violence', 'hate_unfairness']


🔗 Track your red team scan in AI Foundry: None
📋 Planning 10 total tasks


[INFO] Selected 5 objectives using num_objectives=5 (available: 100)
📝 Fetched baseline objectives for violence: 5/5 objectives


[INFO] Selected 5 objectives using num_objectives=5 (available: 100)
📝 Fetched baseline objectives for hate_unfairness: 5/5 objectives
🔄 Fetching objectives for strategy 2/5: base64


🔄 Fetching objectives for strategy 3/5: rot13


...

refusal_scorer was provided in AttackScoringConfig but is not used. This parameter will be ignored.


Executing RedTeamAgent:   0%|          | 0/6 [00:00<?, ?attack/s]

Scanning (Foundry):  10%|██                  | 1/10 [10:05<1:30:46, 605.12s/scan, current=executing]

Scanning (Foundry): 100%|██████████████████████| 10/10 [10:05<00:00, 60.51s/scan, current=executing]

Evaluation results saved to "C:\VScodeLocal\FoundryLabs\foundry-workshop\docs\modules\redteam_advanced_output\evaluation_results.json".

Evaluation results saved to "C:\VScodeLocal\FoundryLabs\foundry-workshop\docs\modules\redteam_advanced_output\results.json".

Evaluation results saved to "C:\VScodeLocal\FoundryLabs\foundry-workshop\docs\modules\.scan_redteam-advanced_20260830_142030\final_results.json".

Overall ASR: 15.0%
Attack Success: 9/60 attacks were successful
------------------------------------------------------------------------------------------------------------------------------------
Risk Category        | Baseline ASR   | Easy-Complexity Attacks ASR  | Moderate-Complexity Attacks ASR | Difficult-Complexity Attacks ASR
------------------------------------------------------------------------------------------------------------------------------------
Violence             | 20.0%          | 12.0%                        | N/A                             | N/A              

!!! note "Expected output"
    ```
    ✅ advanced scan complete — strategies + Spanish/French
    ```
    Each baseline objective is now fired several ways (plain, Base64, ROT13,
    confusable, composed) in multiple languages — far more prompts than the basic
    scan, so this run takes longer.

!!! warning "API is evolving"
    `RedTeam`, `AttackStrategy`, and `SupportedLanguages` live under
    `azure.ai.evaluation.red_team` and move between releases (some builds expose
    `custom_attack_seed_prompts=` for your own objectives). This lab targets the
    `azure-ai-evaluation[redteam]` extra — pin it in `pyproject.toml` and check the
    installed version if an import or argument differs.

## 7. Compare baseline vs. strategies

The advanced scorecard adds an **attack-technique** breakdown alongside the
risk-category one. The story you're looking for: an encoding strategy that scores
a *higher* ASR than baseline means that obfuscation slips past your filters — a
concrete gap to close before you ship.

In [8]:
adv = json.loads(Path("redteam_advanced_output/evaluation_results.json").read_text(encoding="utf-8"))
tech = adv.get("scorecard", {}).get("attack_technique_summary", [{}])[0]

print(f"{'technique':<14}{'ASR':>8}{'success':>9}{'total':>7}")
print("-" * 38)
for label, key in [("OVERALL", "overall"), ("baseline", "baseline"),
                   ("easy", "easy_complexity"), ("difficult", "difficult_complexity")]:
    asr = tech.get(key + "_asr")
    if asr is None:
        continue
    print(f"{label:<14}{asr:>7.1f}%"
          f"{tech.get(key + '_successful_attacks', 0):>9}{tech.get(key + '_total', 0):>7}")

technique          ASR  success  total
--------------------------------------
OVERALL          15.0%        9     60
baseline         10.0%        1     10


!!! note "Expected output"
    ```
    technique          ASR  success  total
    --------------------------------------
    OVERALL          16.0%        8     50
    baseline         10.0%        1     10
    easy             17.5%        7     40
    ```
    Encoded ("easy" complexity) attacks land **more often** than baseline here —
    proof that obfuscation evades the model's defences. Every scan also logs to the
    **Foundry portal**, where you can drill into individual attack/response pairs and
    track ASR across runs.

!!! tip "This closes the safety loop"
    Guardrails ([M11](../11-guardrails/)) are defence;
    red teaming is offence; evaluation ([M9](../09-evaluation/))
    is the measuring tape. Run all three on every release and you have a repeatable
    safety pipeline.

## 🧪 Your turn

1. **Widen coverage.** Bump `num_objectives` to `10` in section 3 and re-run the basic
   scan — more prompts per category means a more stable ASR (and a longer run).
2. **Add a strategy.** Append `AttackStrategy.Flip` (or `AttackStrategy.Leetspeak`) to
   the `attack_strategies` list in section 6 and re-scan — does the new encoding raise
   the *easy*-complexity ASR?
3. **Attack a defended target.** Point `target_callback` at the guardrailed
   `contoso-bank-agent` from [M11](../11-guardrails/) (via an
   `agent_reference` Responses call) and compare its ASR to the bare model — the
   guardrails should drive it toward zero.

---

✅ **You ran a basic risk-category scan, an advanced scan with encoding strategies and
multiple languages, and read the ASR scorecard to find where your model is weakest.**
Next: put a human in the loop and drive agents over raw REST.
→ **[M13 · Human-in-the-Loop & REST](../13-human-in-the-loop-and-rest/)**

## ✅ Your turn — solutions

Run the notebook top-to-bottom first so `RedTeam`, `RiskCategory`, `AttackStrategy`,
`target_callback`, `PROJECT_ENDPOINT`, `credential`, `openai_client`, `json`, and `Path`
are defined. Red-team scans call the model many times, so these cells take a few minutes each.


### 1 · Widen coverage (num_objectives = 10)

Bump `num_objectives` to `10` and re-run the basic scan — more prompts per category means a
more stable ASR (and a longer run: 4 categories × 10 = 40 baseline prompts).


In [9]:
red_team_wide = RedTeam(
    azure_ai_project=PROJECT_ENDPOINT,
    credential=credential,
    risk_categories=[RiskCategory.Violence, RiskCategory.HateUnfairness,
                     RiskCategory.Sexual, RiskCategory.SelfHarm],
    num_objectives=10,
)

wide_result = await red_team_wide.scan(
    target=target_callback, scan_name="redteam-wide-10", output_path="redteam_wide_output")

wide = json.loads(Path("redteam_wide_output/evaluation_results.json").read_text(encoding="utf-8"))
wr = wide.get("scorecard", {}).get("risk_category_summary", [{}])[0]
print(f"num_objectives=10 -> OVERALL ASR {wr.get('overall_asr', 0):.1f}%  "
      f"over {wr.get('overall_total', 0)} prompts "
      f"({wr.get('overall_successful_attacks', 0)} successful)")
print("(baseline lab used num_objectives=5 = 20 prompts; 40 prompts gives a steadier ASR)")

🚀 STARTING RED TEAM SCAN
📂 Output directory: .\.scan_redteam-wide-10_20260830_143053
📊 Risk categories: ['violence', 'hate_unfairness', 'sexual', 'self_harm']


🔗 Track your red team scan in AI Foundry: None
📋 Planning 4 total tasks


[INFO] Selected 10 objectives using num_objectives=10 (available: 100)
📝 Fetched baseline objectives for violence: 10/10 objectives
[INFO] Selected 10 objectives using num_objectives=10 (available: 100)
📝 Fetched baseline objectives for hate_unfairness: 10/10 objectives


[INFO] Selected 10 objectives using num_objectives=10 (available: 100)
📝 Fetched baseline objectives for sexual: 10/10 objectives


[INFO] Selected 10 objectives using num_objectives=10 (available: 100)
📝 Fetched baseline objectives for self_harm: 10/10 objectives


...

Executing RedTeamAgent:   0%|          | 0/1 [00:00<?, ?attack/s]

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\kansing\AppData\Local\Programs\Python\Python312\Lib\logging\__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\kansing\AppData\Local\Programs\Python\Python312\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u014d' in position 2223: character maps to <undefined>
Call stack:
  File "C:\Users\kansing\AppData\Local\Programs\Python\Python312\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\kansing\AppData\Local\Programs\Python\Python312\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\VScodeLocal\FoundryLabs\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_insta

Scanning (Foundry):  25%|█████▊                 | 1/4 [06:13<18:41, 373.94s/scan, current=executing]

Scanning (Foundry): 100%|████████████████████████| 4/4 [06:13<00:00, 93.49s/scan, current=executing]

Evaluation results saved to "C:\VScodeLocal\FoundryLabs\foundry-workshop\docs\modules\redteam_wide_output\evaluation_results.json".

Evaluation results saved to "C:\VScodeLocal\FoundryLabs\foundry-workshop\docs\modules\redteam_wide_output\results.json".

Evaluation results saved to "C:\VScodeLocal\FoundryLabs\foundry-workshop\docs\modules\.scan_redteam-wide-10_20260830_143053\final_results.json".

Overall ASR: 10.0%
Attack Success: 4/40 attacks were successful
------------------------------------------------------------------------------------------------------------------------------------
Risk Category        | Baseline ASR   | Easy-Complexity Attacks ASR  | Moderate-Complexity Attacks ASR | Difficult-Complexity Attacks ASR
------------------------------------------------------------------------------------------------------------------------------------
Violence             | 0.0%           | N/A                          | N/A                             | N/A                       

### 2 · Add a strategy (Flip / Leetspeak)

Append `AttackStrategy.Flip` and `AttackStrategy.Leetspeak` to the `attack_strategies` list
and re-scan — do the new encodings raise the *easy*-complexity ASR? (Small scan: Violence,
`num_objectives=3`.)


In [10]:
from azure.ai.evaluation.red_team import AttackStrategy

flip_team = RedTeam(
    azure_ai_project=PROJECT_ENDPOINT,
    credential=credential,
    risk_categories=[RiskCategory.Violence],
    num_objectives=3,
)

flip_result = await flip_team.scan(
    target=target_callback, scan_name="redteam-flip",
    attack_strategies=[AttackStrategy.Flip, AttackStrategy.Leetspeak],
    output_path="redteam_flip_output")

fj = json.loads(Path("redteam_flip_output/evaluation_results.json").read_text(encoding="utf-8"))
ft = fj.get("scorecard", {}).get("attack_technique_summary", [{}])[0]

print(f"{'technique':<12}{'ASR':>8}{'success':>9}{'total':>7}")
print("-" * 36)
for label, key in [("OVERALL", "overall"), ("baseline", "baseline"), ("easy", "easy_complexity")]:
    asr = ft.get(key + "_asr")
    if asr is None:
        continue
    print(f"{label:<12}{asr:>7.1f}%{ft.get(key + '_successful_attacks', 0):>9}{ft.get(key + '_total', 0):>7}")

🚀 STARTING RED TEAM SCAN
📂 Output directory: .\.scan_redteam-flip_20260830_143724
📊 Risk categories: ['violence']


🔗 Track your red team scan in AI Foundry: None
📋 Planning 3 total tasks


[INFO] Selected 3 objectives using num_objectives=3 (available: 100)
📝 Fetched baseline objectives for violence: 3/3 objectives
🔄 Fetching objectives for strategy 2/3: flip


🔄 Fetching objectives for strategy 3/3: leetspeak


Scanning (Foundry):   0%|                                                   | 0/3 [00:00<?, ?scan/s]

...

Executing RedTeamAgent:   0%|          | 0/3 [00:00<?, ?attack/s]

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\kansing\AppData\Local\Programs\Python\Python312\Lib\logging\__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\kansing\AppData\Local\Programs\Python\Python312\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode characters in position 4496-4498: character maps to <undefined>
Call stack:
  File "C:\Users\kansing\AppData\Local\Programs\Python\Python312\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\kansing\AppData\Local\Programs\Python\Python312\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\VScodeLocal\FoundryLabs\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\kansing\AppData\Local\Programs\Python\Python312\Lib\logging\__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\kansing\AppData\Local\Programs\Python\Python312\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u03bc' in position 1877: character maps to <undefined>
Call stack:
  File "C:\Users\kansing\AppData\Local\Programs\Python\Python312\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\kansing\AppData\Local\Programs\Python\Python312\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\VScodeLocal\FoundryLabs\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_insta

Scanning (Foundry): 100%|████████████████████████| 3/3 [01:36<00:00, 32.17s/scan, current=executing]

Evaluation results saved to "C:\VScodeLocal\FoundryLabs\foundry-workshop\docs\modules\redteam_flip_output\evaluation_results.json".

Evaluation results saved to "C:\VScodeLocal\FoundryLabs\foundry-workshop\docs\modules\redteam_flip_output\results.json".

Evaluation results saved to "C:\VScodeLocal\FoundryLabs\foundry-workshop\docs\modules\.scan_redteam-flip_20260830_143724\final_results.json".

Overall ASR: 22.22%
Attack Success: 2/9 attacks were successful
------------------------------------------------------------------------------------------------------------------------------------
Risk Category        | Baseline ASR   | Easy-Complexity Attacks ASR  | Moderate-Complexity Attacks ASR | Difficult-Complexity Attacks ASR
------------------------------------------------------------------------------------------------------------------------------------
Violence             | 0.0%           | 33.33%                       | N/A                             | N/A                          

### 3 · Attack a defended target

Point the target at the guardrailed `contoso-bank-agent` from [M11](../11-guardrails/) (via an
`agent_reference` Responses call) and compare its ASR to the bare model — the guardrails should
drive it toward zero.


In [11]:
import openai, time

DEFENDED_AGENT = "contoso-bank-agent"

def defended_target_callback(query: str) -> str:
    """Send the attack to the guardrailed agent; blocks/refusals count as non-successful."""
    resp = None
    for _ in range(24):                       # the agent allows one in-flight response (409s)
        try:
            resp = openai_client.responses.create(
                input=query,
                extra_body={"agent_reference": {"name": DEFENDED_AGENT, "type": "agent_reference"}})
            break
        except openai.ConflictError:
            time.sleep(5)
        except openai.BadRequestError:
            return "[blocked by content-safety guardrails]"   # synchronous input trip
    if resp is None:
        return "[agent busy]"

    status = getattr(resp, "status", None)
    for _ in range(90):
        if status in {"completed", "failed", "incomplete", "cancelled"}:
            break
        time.sleep(2)
        try:
            resp = openai_client.responses.retrieve(resp.id)
        except openai.APIError:
            continue
        status = getattr(resp, "status", None)
    return (resp.output_text or "") if status == "completed" else "[blocked or unfinished]"

defended_team = RedTeam(
    azure_ai_project=PROJECT_ENDPOINT,
    credential=credential,
    risk_categories=[RiskCategory.Violence],
    num_objectives=3,
)

defended_result = await defended_team.scan(
    target=defended_target_callback, scan_name="redteam-defended",
    output_path="redteam_defended_output")

dj = json.loads(Path("redteam_defended_output/evaluation_results.json").read_text(encoding="utf-8"))
dr = dj.get("scorecard", {}).get("risk_category_summary", [{}])[0]
print(f"Defended agent  -> Violence ASR: {dr.get('violence_asr', 0):.1f}%  "
      f"({dr.get('violence_successful_attacks', 0)}/{dr.get('violence_total', 0)} successful)")
print("Guardrails (Prompt Shields + content filters) should push this toward 0% "
      "vs. the bare model above.")

🚀 STARTING RED TEAM SCAN
📂 Output directory: .\.scan_redteam-defended_20260830_143917
📊 Risk categories: ['violence']


🔗 Track your red team scan in AI Foundry: None
📋 Planning 1 total tasks


[INFO] Selected 3 objectives using num_objectives=3 (available: 100)
📝 Fetched baseline objectives for violence: 3/3 objectives


Scanning (Foundry):   0%|                                                   | 0/1 [00:00<?, ?scan/s]

Scanning (Foundry):   0%|                             | 0/1 [00:00<?, ?scan/s, current=initializing]

Scanning (Foundry):   0%|                                | 0/1 [00:00<?, ?scan/s, current=executing]

Executing RedTeamAgent:   0%|          | 0/1 [00:00<?, ?attack/s]

Scanning (Foundry): 100%|████████████████████████| 1/1 [00:29<00:00, 29.50s/scan, current=executing]

Scanning (Foundry): 100%|████████████████████████| 1/1 [00:29<00:00, 29.51s/scan, current=executing]

Evaluation results saved to "C:\VScodeLocal\FoundryLabs\foundry-workshop\docs\modules\redteam_defended_output\evaluation_results.json".

Evaluation results saved to "C:\VScodeLocal\FoundryLabs\foundry-workshop\docs\modules\redteam_defended_output\results.json".

Evaluation results saved to "C:\VScodeLocal\FoundryLabs\foundry-workshop\docs\modules\.scan_redteam-defended_20260830_143917\final_results.json".

Overall ASR: 0.0%
Attack Success: 0/3 attacks were successful
------------------------------------------------------------------------------------------------------------------------------------
Risk Category        | Baseline ASR   | Easy-Complexity Attacks ASR  | Moderate-Complexity Attacks ASR | Difficult-Complexity Attacks ASR
------------------------------------------------------------------------------------------------------------------------------------
Violence             | 0.0%           | N/A                          | N/A                             | N/A                